# 8 · Interaction & dashboards

The payoff of a web map is interaction. `hover` customises tooltips; `tap_profile`/`on_tap` respond to
clicks; `draw`/`aoi_crop` let the user draw an area of interest and crop a raster to it; `cross_filter`
links selections across panels. `quickmap(backend="interactive")` is the one-call entry point, and the
dashboard builders (`dashboard`, `layer_control`, `attribute_table`) assemble Panel apps. These are **live**
features — they need a running kernel/server to fire callbacks — so the static images below show the map the
app is built around.

**Setup** — backends and the Rhine gauges + Lisbon DEM.

In [ ]:
import warnings
warnings.filterwarnings("ignore")            # hide backend-mismatch style warnings in the docs

from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs from
# docs/examples/interactive/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"

import io
from IPython.display import Image
import matplotlib.pyplot as plt
import holoviews as hv
hv.extension("bokeh", "matplotlib")          # Bokeh = interactive; matplotlib = static (projection/barbs)

def show(m, static=False, dpi=96):
    """Display the map — **interactive Bokeh** by default (pan / zoom / hover).

    RUN THIS NOTEBOOK (Run All / Shift-Enter) to see the live interactive maps: a saved .ipynb only
    shows interactive plots once the viewer executes their JavaScript (a running kernel, or a trusted
    notebook). Pass ``static=True`` (used for the matplotlib-only layers — projection, streamlines,
    wind barbs, which have no Bokeh glyph) to render a static PNG instead.
    """
    if static or getattr(m, "_projection", None) is not None:
        fig = hv.render(m.render(), backend="matplotlib")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
        plt.close(fig)
        return Image(data=buf.getvalue())
    return m.render()                        # interactive Bokeh object

from pyramids.dataset import Dataset
from pyramids.feature import FeatureCollection
from digitalearth.interactive import InteractiveMap

gauges = FeatureCollection.read_file(str(DATA / "rhine_gauges.geojson"))
dem = Dataset.read_file(str(DATA / "LisbonElevation.tif"))

### `quickmap(backend='interactive')` — one call
The same `quickmap` entry point as the static tier; `backend='interactive'` returns an `InteractiveMap` fed from the same auto-style pipeline.

In [ ]:
from digitalearth import quickmap

m = quickmap(gauges, crs=3857, backend="interactive", basemap=True)
show(m)

### `hover` — custom tooltips
Enable hover with a chosen tooltip list. In the live map, moving the cursor over a gauge shows its name and discharge.

In [ ]:
m = InteractiveMap(crs=3857, title="hover the gauges")
m.points(gauges, column="discharge", size=10, cmap="plasma")
m.hover(tooltips=[("name", "@name"), ("discharge", "@discharge")])
show(m)

### `draw` + `aoi_crop` — draw an area of interest
`draw('box')` adds a box-draw tool; in a live kernel you draw a rectangle and `aoi_crop(dataset)` clips a pyramids raster to it.

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="draw a box to crop")
m.field(dem, cmap="terrain").draw("box")
show(m)

### `layer_control` — a layer panel
`layer_control` builds a Panel app with per-layer toggles, an opacity slider and a basemap switch, keeps it on the map, and returns the map so the call chains; read the app back from `layer_control_panel` (live only). The static image shows the composed map.

In [ ]:
m = InteractiveMap(crs=3857, title="layer control")
m.basemap("CartoLight").points(gauges, column="discharge", size=9, cmap="plasma")
app = m.layer_control().layer_control_panel   # the Panel app it built (live only)
show(m)

### `attribute_table` — linked table
`attribute_table` shows the gauge attributes as a Panel table linked to the map (selecting rows highlights features in a live session).

In [ ]:
m = InteractiveMap(crs=3857)
m.points(gauges, column="discharge", size=8, cmap="plasma")
table = m.attribute_table(gauges)   # a Panel table (live only)
show(m)